# Evaluación de la traducción

1. Se tiene que hacer un filtrado y una limpieza por modelo. 
2. La limpieza automática se tiene que compensar con un filtrado manual, ya que los modelos NO SIEMPRE SIGUEN BIEN LAS INSTRUCCIONES.
3. Hay que medir los valores donde funcionan mal para reportarlo.

In [14]:
import pandas as pd
import re

def list_to_str(qwen_list):
    text = ''
    for value in qwen_list:
        new_val = re.sub('  ', '', value)
        text += new_val + '\n'
    return text

def get_trans(path, test):
    if test:
        start = 226 
    else:
        start = 203
    dataset = pd.read_csv(path)
    dataset = dataset.drop(columns = ["Unnamed: 0"])
    print("Longitud del dataset: {}".format(len(dataset)))
    trans = dataset[:start]
    return trans

folio_test = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl', lines = True)
folio_val = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_validation.jsonl', lines = True)

trans_test = get_trans('/home/flopezp/Kurosagol/Ongoing/baseline_datasets/test/DeepSeek-R1-0528-Qwen3-8B_full.csv', True)
print("Longitudes de los subsplits:", len(trans_test))

trans_val = get_trans('/home/flopezp/Kurosagol/Ongoing/baseline_datasets/validation/DeepSeek-R1-0528-Qwen3-8B_full.csv', False)
print("Longitudes de los subsplits:", len(trans_val))

Longitud del dataset: 678
Longitudes de los subsplits: 226
Longitud del dataset: 609
Longitudes de los subsplits: 203


In [15]:
# LogicSim+
def get_nice_dict(regex, instance, const):
	"""
		A partir de una regex, extrae todos los valores encontrados y los formatea para tener un diccionario con apariciones.

		regex = str ; Expresión regular a buscar.
		const = bool ; Determina si estamos analizando constantes. Esto permite que se realice un filtro extra en el código.
	"""
	lista = []
	for _ in instance:
		aux = re.finditer(regex, _)
		for expression in aux:
			if const:
				regex_list = expression.group()[1:-1]
				regex_list = regex_list.split(',')
				for j in regex_list:
					if len(j) > 1:
						constant = re.sub(' ', '', j)
						lista.append(constant)
			else:
				lista.append(expression.group())

	set_set = list(set(lista))
	dict_dict = {}
	for _ in set_set:
		dict_dict["{}".format(_)] = lista.count(_)

	return dict_dict


def extract_info(ds_answer, llm):
	"""
		Limpia la respuesta de un dataset y extrae las premisas necesarias para calcular LogicSim.

		llm = Bool ; Señala si se va a evaluar la respuesta generada por un LLM.
	"""
	instance = ds_answer.split('\n')
	
    # Este bloque permite limpiar las respuestas que siguen en exceso el prompt inicial.
	if llm: 
		for i in range(len(instance)):
			instance[i] = re.sub('(::)+([ A-z.]+)', '', instance[i])
			instance[i] = re.sub('(:::)+([ A-z.]+)', '', instance[i])
			instance[i] = re.sub('(  )+', '', instance[i])

	pred_dict = get_nice_dict(r'[A-z]+\(([A-z]+(,? [A-z]+)*)\)', instance, False)
	const_dict = get_nice_dict(r'(\([A-z]+(\, [A-z]+){0,}\))', instance, True)
	logop_dict = get_nice_dict(r'[∀∧→⊕¬∨∃]+', instance, False)

	# len(instance) = Cantidad de premisas
	# pred_dict = Diccionario con cantidad de predicados. DE AQUÍ SE OBTIENE APARICIONES TOTALES Y CANTIDAD DE PREDICADOS
	# constant_dict =  Diccionario con cantidad de constantes. DE AQUÍ SE OBTIENE APARICIONES TOTALES Y CANTIDAD DE CONSTANTES
	# logop_dict =  Diccionario con cantidad de operadores y cuantificadores. DE AQUÍ SE OBTIENE APARICIONES TOTALES Y CANTIDAD DE CUANTIFICADORES
	return pred_dict, const_dict, logop_dict, len(instance)


def total_apps(dict1, preds):
    """
        Obtiene:
            1) La cantidad total de apariciones de un valor (Const/Pred/LogOps)
            2) La cantidad de constantes/predicados/logops distintos.

        dict1 = dict ; El diccionario obtenido previamente.
        preds = bool ; Valor booleano que permite realizar un filtro sobre los predicados.
    """
    if preds:
        lista_chida_aux = [re.search(r'([A-z]+\()', _).group() for _ in dict1.keys()]
        lista_chida = list(set(lista_chida_aux))

        # Guardamos la cantidad de apariciones en el diccionario previo.
        aux_dict = {"{}".format(_):0 for _ in lista_chida}
        for key in dict1.keys():
            for _ in lista_chida:
                if _ in key:
                    aux_dict["{}".format(_)] += dict1[key]
        
        if preds:
            aux_dict = {"{}".format(_[:-1]): aux_dict[_] for _ in aux_dict}
    else:
        aux_dict = dict1
    
    total_value = 0
    for _ in aux_dict:
        total_value += aux_dict[_]

    #total_value = Apariciones totales
    #len(aux_dict) = Cantidad de [VALUE] distinto.
    return total_value, len(aux_dict)


def logic_sim_plus_individual(llm_ans, folio_ans):
    """
        Extrae los valores distintos y valores totales de cada instancia. 
    """
    pred_llm, const_llm, logops_llm, len_llm = extract_info(llm_ans, True)
    pred_folio, const_folio, logops_folio, len_folio = extract_info(folio_ans, False)

    #llm
    dif_preds_llm, pred_count_llm = total_apps(pred_llm, True)
    dif_const_llm, const_count_llm = total_apps(const_llm, False)
    dif_logops_llm, logops_count_llm = total_apps(logops_llm, False)

    #folio
    dif_preds_folio, pred_count_folio = total_apps(pred_folio, True)
    dif_const_folio, const_count_folio = total_apps(const_folio, False)
    dif_logops_folio, logops_count_folio = total_apps(logops_folio, False)

    #Absolute Values
    dif_preds = abs(dif_preds_llm - dif_preds_folio)
    tot_aps_preds = abs(pred_count_llm - pred_count_folio)
    
    dif_const = abs(dif_const_llm - dif_const_folio)
    tot_aps_const = abs(const_count_llm - const_count_folio)

    dif_logops = abs(dif_logops_llm - dif_logops_folio)
    tot_aps_logops = abs(logops_count_llm - logops_count_folio)

    dif_premises = abs(len_llm - len_folio)

    logicsim_plus = dif_preds + tot_aps_preds + dif_const + tot_aps_const + dif_logops + tot_aps_logops + dif_premises
    return logicsim_plus

### Filtros DeepSeek

In [16]:
# Pruebas para deepseek.
# Limpieza que obtiene las premisas solas. Se tiene que complementar con una posible limpieza manual.
full_list = []
for i in range(len(trans_test)):
    text = trans_test["Full"].iloc[i]
    aux = text.split('---')[0]
    split_vals = aux.split('\n')[:-1]
    for elem in split_vals:
        three_dots = re.search('(:::)', elem)
        if three_dots != None:
            new_val = elem.split(':::')[0]
            new_val = re.sub(r'(  )+|\t', '', new_val)
            split_vals[split_vals.index(elem)] = new_val
    #print(split_vals)
    full_list.append(split_vals)

    #print(i, len(split_vals), split_vals)
    #print("=============")
print(len(full_list))

226


In [17]:
"""
for elem in full_list[20]:
   search = re.search("(FOL Premises)", elem)
   if search != None:
      value = full_list[20].index(elem)

new_list = full_list[20][value+1:]
print(new_list)
"""

def clean_extra(instance):
   """
      Filtra las listas que tienen mucho texto además de las premisas.
   """
   try:
      for elem in instance:
         search = re.search("(FOL Premises)", elem)
         if search != None:
            val = instance.index(elem)
      new_list = instance[val+1:]
      return new_list
   except:
      return instance

In [18]:
for elemento in full_list:
    if len(elemento) > 10:
        index = full_list.index(elemento)
        clean_list = clean_extra(elemento)
        full_list[index] = clean_list

print(len(full_list))
        

226


In [19]:
full_list

[['∀x (ProfessionalSoccerPlayer(x) → ¬ProfessionalBasketballPlayer(x)) ',
  '∀x (ProfessionalSoccerDefender(x) → ProfessionalSoccerPlayer(x)) ',
  '∀x (ProfessionalCenterback(x) → ProfessionalSoccerDefender(x)) ',
  '∃x (Athlete(x) ∧ ProfessionalCenterback(x)) ',
  'ProfessionalBasketballPlayer(stephenCurry) '],
 ['∀x (ProfessionalSoccerPlayer(x) → ¬ProfessionalBasketballPlayer(x)) ',
  '∀x (ProfessionalSoccerDefender(x) → ProfessionalSoccerPlayer(x)) ',
  '∀x (ProfessionalCenterback(x) → ProfessionalSoccerDefender(x)) ',
  '∃x (Athlete(x) ∧ ProfessionalCenterback(x)) ',
  'ProfessionalBasketballPlayer(stephenCurry) '],
 ['∀x (ProfessionalSoccerPlayer(x) → ¬ProfessionalBasketballPlayer(x)) ',
  '∀x (ProfessionalSoccerDefender(x) → ProfessionalSoccerPlayer(x)) ',
  '∀x (ProfessionalCenterback(x) → ProfessionalSoccerDefender(x)) ',
  '∃x (Athlete(x) ∧ ProfessionalCenterback(x)) ',
  'ProfessionalBasketballPlayer(stephenCurry) '],
 ['∀x (JoinSG(x) → ImproveComm(x)) ',
  '∀x (StayAlone(x) 

### Filtros Qwen

In [6]:
# Filtrado para qwen3-14b
first_clean = []

for i in range(len(trans_test)):
    ds_instance = trans_test["Full"][i]
    think = re.search(r'</think>', ds_instance)
    fol_p = re.search(r"FOL Premises:", ds_instance)
    nl_p = re.search(r"Natural Language Premises:", ds_instance)
    if think != None:
        first_clean.append(ds_instance[think.end():])
    
    elif re.search(r'(----)+', ds_instance):
        split = trans_test["Full"][i].split('--------------')[0]
        first_clean.append(split)

    elif fol_p != None:
        first_clean.append(ds_instance[:fol_p.start()])
    
    elif nl_p != None:
        first_clean.append(ds_instance[:nl_p.start()])
    
    print("-.-.-...-.-.-.-.-.--.-.-.-")

-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-.-.-...-.-.-.-.-.--.-.-.-
-

In [ ]:
for i in range(len(trans)):
    try:
        split = re.search('--------------', trans["Full"][i])
        print(len(trans["Full"][i]) - split.end())
    except:
        print("No hay --------------")

#Ojo: Tal vez no tenemos que dar tanto punto contexto para la generación. Siento que 7000-9000 es demasiado.

print("=============== Cambio a LogicSim+ ===============")

for i in range(len(trans)):
    clear_val = trans["Full"][i].split('--------------')[0].split('\n')[:-1] # Esto es un problema si nunca sale el '--------------'
    string = list_to_str(clear_val)
    fol = folio_full["premises-FOL"][i]
    print(logic_sim_plus_individual(string, fol))

# Hay una situación con el filtrado de información. La separación no siempre es tan constante. Tenemos que analizar caso por caso, o encontrar la forma de separar bien. 

20427
17496
17496
22620
22620
14623
14623
14623
73
18800
18800
18800
517
No hay --------------
No hay --------------
19284
19284
19284
19275
19275
19275
15515
18902
No hay --------------
No hay --------------
18934
18934
18934
18934
18934
18934
18934
18934
14985
16514
15714
437
437
151
151
17482
18826
18826
18826
18826
18826
18683
17021
17021
17021
17021
No hay --------------
No hay --------------
15982
15982
15982
16867
16974
16413
18073
18073
18073
18073
18073
18491
18491
18491
15899
16273
18130
18130
395
395
395
No hay --------------
No hay --------------
No hay --------------
20709
20709
19719
No hay --------------
No hay --------------
No hay --------------
9446
9446
9446
9446
18435
18435
18435
18959
No hay --------------
18985
18985
18985
18985
18985
4466
4466
4466
17900
17900
19081
No hay --------------
No hay --------------
19170
19170
19170
16021
16021
16021
1055
1055
22155
18471
No hay --------------
No hay --------------
No hay --------------
No hay --------------
19741
1974

**LO QUE SIGUE ES LA IMPLEMENTACIÓN CON DATASETS VIEJOS.**

Se pueden reutilizar partes, pero es necesario ajustarlo a los nuevos datasets.

In [ ]:
from datasets import load_dataset
import pandas as pd
import re
import os
from nltk.inference.prover9 import *

# Cluster de Helena. Bendita sea la doctora en verdad.
os.environ['PROVER9'] = '/media/discoexterno/francisco/Prover9/bin'


def prove(argument):
    goal, assumptions = argument
    g = Expression.fromstring(goal)
    alist = [Expression.fromstring(a) for a in assumptions]
    p = Prover9Command(g, assumptions=alist).prove()
    return p

def get_full_ds(model):
    """
    model = str ; Debe de ser el nombre del LLM en cuestión: gpt, qwen, nemo, etc etc
    """
    ds1 = pd.read_csv(f'/home/flopezp/Kurosagol/Ongoing/datasets_new/{model}_trans.csv')
    ds2 = pd.read_csv(f'/home/flopezp/Kurosagol/Ongoing/datasets_new/{model}_infer.csv')
    ds3 = pd.read_csv(f'/home/flopezp/Kurosagol/Ongoing/datasets_new/{model}_retrans.csv')

    trans = ds1["Translation"]
    infer = ds2["Inference"]
    retrans = ds3["Retranslation"]

    full_ds = pd.DataFrame({"Translation": trans, "Inference": infer, "Retrans": retrans})
    return full_ds

# Amoxtli
full_qwen = get_full_ds('qwen')
full_gpt = get_full_ds('gpt')

full_qwen.head()

folio_full = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl', lines = True)
folio_full.head()

#new_folio = load_dataset('fol-autoformalization/FOLIO', token = hf_token)
#new_folio

# Server Dra. Helena (bendiciones)
#tran_ds = pd.read_csv(r'/media/discoexterno/francisco/Kurosagol/Ongoing/datasets/gpt_qwen_translations_8ktokens.csv')
#tran_ds = tran_ds.drop(columns=["Unnamed: 0"])
#tran_ds.head()

,story_id,premises,premises-FOL,conclusion,conclusion-FOL,label,example_id
0,324,No professional soccer players play profession...,∀x ((Professional(x) ∧ SoccerPlayer(x)) → ¬Pla...,Stephen Curry is an athlete.,Athlete(stephenCurry),Uncertain,828
1,324,No professional soccer players play profession...,∀x ((Professional(x) ∧ SoccerPlayer(x)) → ¬Pla...,Stephen Curry is an athlete and a professional...,Athlete(stephenCurry) ∧ Professional(stephenCu...,False,829
2,324,No professional soccer players play profession...,∀x ((Professional(x) ∧ SoccerPlayer(x)) → ¬Pla...,"If Stephen Curry is a professional centerback,...",(Professional(stephenCurry) ∧ CenterBack(steph...,True,830
3,357,If people join student government in high scho...,"∀x (Join(x, studentGovernment) → ManyOpportuni...",Tyler joined the student government in high sc...,"JoinIn(tyler, studentGovernment, highSchool)",Uncertain,947
4,357,If people join student government in high scho...,"∀x (Join(x, studentGovernment) → ManyOpportuni...",If Tyler either both stays alone in high schoo...,"¬(StayAloneIn(tyler, highSchool) ⊕ (Well-adjus...",True,948


In [ ]:
print("==================")
print("====始めましょう====")
print("==================")

gpt_vs_qwen = []
gpt_vs_folio = []
qwen_vs_folio = []

for i in range(len(full_qwen)):
    gpt = full_gpt["Translation"][i]
    qwen = full_qwen["Translation"][i]
    folio = folio_full["premises-FOL"][i]

    gptqwen = logic_sim_plus_individual(gpt, qwen)
    gptfolio = logic_sim_plus_individual(gpt, folio)
    qwenfolio = logic_sim_plus_individual(qwen, folio)
    print("GPT vs Qwen: {}".format(gptqwen))
    print("GPT vs FOLIO: {}".format(gptfolio))
    print("FOLIO vs Qwen: {}".format(qwenfolio))

    if gptqwen < 6:
        print("GPT vs QWEN < 6")
        print('-------')
        print(gpt)
        print('-------')
        print(qwen)
        print('-------')
        print(folio)
        print("================")
    
    if gptfolio > 25 or qwenfolio > 25:
        print("LLM vs FOLIO > 25")
        print('-------')
        print(gpt)
        print('-------')
        print(qwen)
        print('-------')
        print(folio)
        print("================")

    gpt_vs_qwen.append(gptqwen)
    gpt_vs_folio.append(gptfolio)
    qwen_vs_folio.append(qwenfolio)
    print('-------------------')

====始めましょう====
GPT vs Qwen: 2
GPT vs FOLIO: 16
FOLIO vs Qwen: 18
GPT vs QWEN < 6
-------
∀x (ProfessionalSoccerPlayer(x) → ¬ProfessionalBasketballPlayer(x))
∀x (ProfessionalSoccerDefender(x) → ProfessionalSoccerPlayer(x))
∀x (ProfessionalCenterBack(x) → ProfessionalSoccerDefender(x))
∃x (Athlete(x) ∧ ProfessionalCenterBack(x))
ProfessionalBasketballPlayer(stephenCurry)
-------


∀x (ProfessionalSoccerPlayer(x) → ¬PlaysBasketball(x))  
∀x (ProfessionalSoccerDefender(x) → ProfessionalSoccerPlayer(x))  
∀x (ProfessionalCenterback(x) → ProfessionalSoccerDefender(x))  
∃x (Athlete(x) ∧ ProfessionalCenterback(x))  
PlaysBasketball(StephenCurry)
-------
∀x ((Professional(x) ∧ SoccerPlayer(x)) → ¬Play(x, professionalBasketball))
∀x ((Professional(x) ∧ Defender(x)) → (Professional(x) ∧ SoccerPlayer(x)))
∀x ((Professional(x) ∧ CenterBack(x)) → (Professional(x) ∧ Defender(x)))
∃x (Athlete(x) ∧ Professional(x) ∧ CenterBack(x))
Play(stephenCurry, professionalBasketball)
-------------------
GPT vs Q

In [ ]:
import statistics

print("LogicSim promedio (GPT vs Qwen): {}".format(round(sum(gpt_vs_qwen)/len(gpt_vs_qwen), 3)))
print("Desviación estándar: {}".format(round(statistics.stdev(gpt_vs_qwen), 3)))
print('----')
print("LogicSim promedio (GPT vs FOLIO): {}".format(round(sum(gpt_vs_folio)/len(gpt_vs_folio), 3)))
print("Desviación estándar: {}".format(round(statistics.stdev(gpt_vs_folio), 3)))
print('----')
print("LogicSim promedio (FOLIO vs Qwen): {}".format(round(sum(qwen_vs_folio)/len(qwen_vs_folio), 3)))
print("Desviación estándar: {}".format(round(statistics.stdev(qwen_vs_folio), 3)))

LogicSim promedio (GPT vs Qwen): 11.704
Desviación estándar: 13.428
----
LogicSim promedio (GPT vs FOLIO): 19.389
Desviación estándar: 13.347
----
LogicSim promedio (FOLIO vs Qwen): 23.279
Desviación estándar: 14.931


In [ ]:
# FOL to Prover9 Syntax

def switch_quantifiers(text, cuantifier):
    """
        Elimina todos los cuantificadores y los traduce a sintaxis de Prover9. Sin importar la variable ni la cantidad de apariciones. Qué pedo soy una verga para esto.

        text = str ;  Texto a modificar.
        cuantifier = str ('forall', 'exists') ; Cuantificador a modificar.
    """
    if cuantifier == 'forall':
        regex = '∀[A-z]'
        cuant = 'all '
    else:
        regex = '∃[A-z]'
        cuant = 'exists '

    owo = re.finditer(regex, text)
    aux_list = list(owo)
    if len(aux_list) == 0:
        return text

    # Redefinimos el iterador porque hacer lista de un iterador lo consume. CHINGA TU MADRE PYTHON. VETE A LA BURGER.    
    owo = re.finditer(regex, text)
    temporal_str = ''

    for _ in owo:
        if temporal_str == '':
            temporal_str = text[0:_.start()] + cuant + _.group()[-1] + '.' + text[_.end():]
        else:
            value = re.search(regex, temporal_str)
            temporal_str = temporal_str[0:(value.start())] + cuant + value.group()[-1] + '.' + temporal_str[value.end():]
    
    return temporal_str



def fol_to_prover9(value):
    """
        Modifica los símbolos lógicos normales y los cambia por los valores adecuados para Prover9.*
    """
    temp = value.lower()
    temp = switch_quantifiers(temp, 'forall')
    temp = switch_quantifiers(temp, 'exists')
    temp = re.sub('-', '', temp)
    temp = re.sub('¬', '-', temp) 
    temp = re.sub('→|→', '-> ', temp)
    temp = re.sub('∧', '&', temp)
    temp = re.sub('∨', '|', temp)
    temp = re.sub('↔', '<->', temp)
    temp = re.sub('≠', '!=', temp)
    temp = re.sub(r'\'', '', temp)
    temp = re.sub(r'\[', '(', temp)
    temp = re.sub(r'\]', ')', temp)
    temp = re.sub('∴', '', temp)
    temp = re.sub('(\. \()', '.(', temp)
    temp = re.sub('\.{2}', '.', temp)
    temp = re.sub(r'\)\.', ')', temp)
    temp = re.sub(r'([^a-z]\.\()', '(', temp)
    return temp


def dash_predicates(text):
    """
        Cambia los predicados de la forma "texto-texto-texto(x)" -> "textotextotexto(x)"

        text = str ; el hilo a modificar.
    """
    all_values = len(re.findall(r'[a-z0-9]+(\-[a-z0-9]+\-{0,})+[a-z0-9]+', text))

    if all_values == 0:
        return text
    
    new_text = text
    for i in range(all_values):
        current_regex = re.search(r'[a-z0-9]+(\-[a-z0-9]+\-{0,})+[a-z0-9]+', new_text)
        split = current_regex.group().split()
        aux_text = ''
        for elem in split:
            aux_text = aux_text + elem
        new_text = new_text[:current_regex.start()] + aux_text + new_text[current_regex.end():]

    return new_text


def elim_spaces(text):
    """
        Elimina los espacios entre variables: lionel messi -> lionelmessi

        text = str; El texto a modificar.


        OBS: Este formato de funciones (Encontrar cantidades y luego iterar sobre las cantidades) me gusta bastante.
    """
    total_iters = len(list(re.finditer('([A-z]+ )+([A-z]{2,})', text)))
    if total_iters == 0:
        return text

    new_text = text
    for i in range(total_iters):
        current_regex = re.search('([A-z]+ )+([A-z]{2,})', new_text)
        split = current_regex.group().split()
        aux_text = ''
        for elem in split:
            aux_text = aux_text + elem
        new_text = new_text[:current_regex.start()] + aux_text + new_text[current_regex.end():]

    return new_text


# El XOR me tiene hasta los huevos cabrón te lo juro.
def xor_bonito(expression):
    """
        Elimina el símbolo de XOR, y lo reescribe en la fórmula (A OR B) AND NOT(A AND B)
    """
    individual_values = re.findall('([A-z|_|0-9]{2,}|¬)', expression)
    a = individual_values[0] + '(x)'
    b = individual_values[1] + '(x)'
    a_or_b = '(' + a + ' | ' + b + ')'
    not_a_and_b = '-(' + a + ' & ' + b +')'
    xor = a_or_b + ' & ' + not_a_and_b
    return xor

def xor_bonito_extreme(expression):
    """
        Elimina los XOR de fórmulas compuestas.
    """
    aux2 = re.search(r'([a-z]+\([a-z, ]+\)) ⊕ ([a-z]+\([a-z, ]+\))', expression)
    split = aux2.group().split('⊕')
    a_f = split[0]
    b_f = split[-1]
    a_or_b = '(' + a_f + ' | ' + b_f + ')'
    not_a_and_b = '-(' + a_f + ' & ' + b_f +')'
    xor = a_or_b + ' & ' + not_a_and_b
    return xor

def rewrite_xor(re_search, element, comp):
    """
        Genera una nueva expresión a partir del xor bonito. 

        re_search = re.search(regex, str)
        element = str ; same str as above
        comp = bool ; True iff predicates have multiple variables.
    """
    start = re_search.start()
    end = re_search.end()

    xor_substr = element[start:end]
    if comp:
        xor_chido = xor_bonito_extreme(xor_substr)
    else:
        xor_chido = xor_bonito(xor_substr)

    nuevo = element[:start] + xor_chido + element[end:]
    return nuevo


def clean(value):
    """
        Procesa una respuesta individual de FOLIO/GPT_TRANS/QWEN_TRANS para que se pase al formato de Prover9.
    """

    clean_premises_aux = value.split('\n')
    clean_premises = []
    for _ in clean_premises_aux:
        if _ != '':
            clean_premises.append(_)

    if "Premises" in clean_premises[0]:
        del clean_premises[0]

    for _ in clean_premises:
        clean_premises[clean_premises.index(_)] = re.sub('(:::)+([ A-z.]+)', '', _)

    for _ in clean_premises:
        clean_premises[clean_premises.index(_)] = re.sub('[0-9]\.', '', _)

    for instance in clean_premises:
        clean_premises[clean_premises.index(instance)] = fol_to_prover9(instance)

    for instance in clean_premises:
        clean_premises[clean_premises.index(instance)] = elim_spaces(instance)

    try:
        # Filtro XOR sencillo
        for element in clean_premises:
            xor_count = len(re.findall('⊕', element))
            if xor_count > 0:
                value = element
                while xor_count > 0:
                    aux = re.search('(-{0,1}[a-z]+\([a-z, ]+\)) ⊕ (-{0,1}[a-z]+\([a-z, ]+\))', element)
                    element = rewrite_xor(aux, element, False)
                    xor_count = len(re.findall('⊕', element))
                clean_premises[clean_premises.index(value)] = element

        # Filtro XOR multipremisa
        for element in clean_premises:
            xor_count = len(re.findall('⊕', element))
            if xor_count > 0:
                value = element
                while xor_count > 0:
                    aux = re.search('(-{0,1}[a-z]+\([a-z, ]+\)) ⊕ (-{0,1}[a-z]+\([a-z, ]+\))', element)
                    element = rewrite_xor(aux, element, True)
                    xor_count = len(re.findall('⊕', element))
                clean_premises[clean_premises.index(value)] = element

    except:
        print("Hubo un Xor malo")    

    return clean_premises

<>:55: SyntaxWarning: invalid escape sequence '\.'
<>:56: SyntaxWarning: invalid escape sequence '\.'
<>:175: SyntaxWarning: invalid escape sequence '\.'
<>:190: SyntaxWarning: invalid escape sequence '\('
<>:201: SyntaxWarning: invalid escape sequence '\('
<>:55: SyntaxWarning: invalid escape sequence '\.'
<>:56: SyntaxWarning: invalid escape sequence '\.'
<>:175: SyntaxWarning: invalid escape sequence '\.'
<>:190: SyntaxWarning: invalid escape sequence '\('
<>:201: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_63037/989746200.py:55: SyntaxWarning: invalid escape sequence '\.'
  temp = re.sub('(\. \()', '.(', temp)
/tmp/ipykernel_63037/989746200.py:56: SyntaxWarning: invalid escape sequence '\.'
  temp = re.sub('\.{2}', '.', temp)
/tmp/ipykernel_63037/989746200.py:175: SyntaxWarning: invalid escape sequence '\.'
  clean_premises[clean_premises.index(_)] = re.sub('[0-9]\.', '', _)
/tmp/ipykernel_63037/989746200.py:190: SyntaxWarning: invalid escape sequence '\('
  aux = re

In [ ]:
for i in range(len(full_qwen)):
    print("=============================")
    print("============={}==============".format(i))
    print("=============================")
    print(folio_full['premises-FOL'][i])
    print('-----------------')
    print(clean(folio_full['premises-FOL'][i]))
    print(clean(full_qwen['Translation'][i]))
    print(clean(full_gpt['Translation'][i]))

=============0==============
∀x ((Professional(x) ∧ SoccerPlayer(x)) → ¬Play(x, professionalBasketball))
∀x ((Professional(x) ∧ Defender(x)) → (Professional(x) ∧ SoccerPlayer(x)))
∀x ((Professional(x) ∧ CenterBack(x)) → (Professional(x) ∧ Defender(x)))
∃x (Athlete(x) ∧ Professional(x) ∧ CenterBack(x))
Play(stephenCurry, professionalBasketball)
-----------------
['all x.((professional(x) & soccerplayer(x)) ->  -play(x, professionalbasketball))', 'all x.((professional(x) & defender(x)) ->  (professional(x) & soccerplayer(x)))', 'all x.((professional(x) & centerback(x)) ->  (professional(x) & defender(x)))', 'exists x.(athlete(x) & professional(x) & centerback(x))', 'play(stephencurry, professionalbasketball)']
['all x.(professionalsoccerplayer(x) ->  -playsbasketball(x))  ', 'all x.(professionalsoccerdefender(x) ->  professionalsoccerplayer(x))  ', 'all x.(professionalcenterback(x) ->  professionalsoccerdefender(x))  ', 'exists x.(athlete(x) & professionalcenterback(x))  ', 'playsbasketb

In [ ]:
def check_arg_validity(index, llm):
    """
        Dadas premisas en lenguaje lógico, verifica que la conclusión correspondiente sea inferible.
    """
    llm_valid, llm_invalid, fol_valid, fol_invalid = 0, 0, 0, 0
    if llm == 'qwen':
        llm_ds = full_qwen
    else:
        llm_ds = full_gpt
    llm_value = llm_ds['Translation'.format(llm)][index]
    folio_value = folio_full['premises-FOL'][index]

    true_validity = folio_full['label'][index]
    print("Logical Validity: {}".format(true_validity))

    clean_llm = clean(llm_value)
    clean_folio = clean(folio_value)
    clean_conc = clean(folio_full['conclusion-FOL'][index])
    
    args_llm = (
        clean_conc[0],
        clean_llm
    )

    args_folio = (
        clean_conc[0],
        clean_folio
    )
    
    # Lo que hacemos aquí es evaluar una conclusión contra las premisas de un LLM o del conjunto de datos.
    # OBS: A veces la premisa NO debe de ser deducible. 
    # ¿Cómo vamos a cuantificar este show?
    # Casos:
    # Premisas: TRUE, UNCERTAIN, FALSE.
    # Posibles resultados: TRUE, FALSE. ¿Cómo lidiamos con Uncertain? -> En estos casos Prover9 te regresa False. 
    
    print("LLM ({}):".format(llm))
    try:
        print("Ejecutable. Valor: {}".format(prove(args_llm)))
        llm_valid += 1
    except Exception as e:
        print(e)
        print(type(e))
        llm_invalid += 1
        
    print("FOLIO:")
    try:
        print("Ejecutable. Valor: {}".format(prove(args_folio)))
        fol_valid += 1
    except Exception as e:
        print(e)
        print(type(e))
        fol_invalid += 1
    print("======================")

    return llm_valid, llm_invalid, fol_valid, fol_invalid
    

**ACHTUNG** ESTO SE TIENE QUE CORRER EN EL CLUSTER DE HELENAAAAAAAAAAAAAAAAAAAAAA QUE NECESITAMOS A PROVER9 JIJIJIJUJUJUJU XD

In [ ]:
_llm_valid, _llm_invalid, _fol_valid, _fol_invalid = 0,0,0,0
for i in range(len(full_qwen)):
    print(i)
    try:
        a, b, c, d = check_arg_validity(i, 'qwen')
        _llm_valid += a
        _llm_invalid += b
        _fol_valid += c
        _fol_invalid += d
    except:
        print("oops")

print("Premisas de LLM válidas: {}".format(_llm_valid))
print("Premisas de LLM inválidas: {}".format(_llm_invalid))
print("Premisas de FOLIO válidas: {}".format(_fol_valid))
print("Premisas de FOLIO inválidas: {}".format(_fol_invalid))


0
Logical Validity: Uncertain
LLM (qwen):


NLTK was unable to find the prover9 file!
Use software specific configuration parameters or set the PROVER9 environment variable.

  Searched in:
    - /usr/local/bin/prover9
    - /usr/local/bin/prover9/bin
    - /usr/local/bin
    - /usr/bin
    - /usr/local/prover9
    - /usr/local/share/prover9

  For more information on prover9, see:
    <https://www.cs.unm.edu/~mccune/prover9/>
<class 'LookupError'>
FOLIO:


NLTK was unable to find the prover9 file!
Use software specific configuration parameters or set the PROVER9 environment variable.

  Searched in:
    - /usr/local/bin/prover9
    - /usr/local/bin/prover9/bin
    - /usr/local/bin
    - /usr/bin
    - /usr/local/prover9
    - /usr/local/share/prover9

  For more information on prover9, see:
    <https://www.cs.unm.edu/~mccune/prover9/>
<class 'LookupError'>
1
Logical Validity: False
LLM (qwen):


NLTK was unable to find the prover9 file!
Use software specific configuration parameters 

In [ ]:
# Vamos a tomarnos una pausa de la traducción a la sintaxis de Prover9.
# Todavía falta por ver unos casos tipo: e(x) y mejorar el hdspm del XOR.
# Ahora vamos a generar las respuestas del modelo para el proceso de inferencia.